# Do the monthly loan balances improve the model?

This notebook adds the 12 features from `09_pos_cash_eda_features.ipynb` to the model built in `08_model_with_installments.ipynb`, and measures whether the model gets better.

## Where things stand

| Model | Features | Cross-validation | Kaggle private |
|---|---:|---:|---:|
| Application only (notebook 02) | 36 | 0.7685 | 0.7638 |
| With the credit bureau (notebook 04) | 60 | 0.7779 | 0.7753 |
| With earlier applications (notebook 06) | 85 | 0.7856 | 0.7842 |
| With the payment record (notebook 08) | 108 | 0.7932 | 0.7934 |

The model to beat is the last one.

## The expectation, written in notebook 09 before this notebook was started

A gain at or below the noise floor. The table describes the same loans as the payment record and `previous_application`. Its arrears are a coarser view of lateness the payment record already gives, and its most promising summary turned out to be a feature notebook 05 had already built. Of 21 candidates, 12 survived screening, and the largest signal beyond the current model was 0.522.

Notebook 09 also recorded that this screen has been wrong before, in both directions. That is why the test is run.

## What counts as an improvement

The four conditions of notebooks 04, 06 and 08, unchanged. On the same five development folds, against the current model: the mean ROC AUC is higher; higher in at least four of five folds; the gain is more than twice its standard error; and the gain exceeds the seed noise floor (0.0005, or larger if stage 2 measures more).

## What this notebook does, and when it stops

| # | Stage | Question it answers |
|---|---|---|
| 1 | Setup, join and audit | Do the five feature tables join exactly, and is the split unchanged? |
| 2 | Baseline | Does the current 108-feature model still score 0.7932 when rebuilt here? What is the noise floor? |
| 3 | Add the new features | With every setting unchanged, what is the gain, and does it meet the four conditions? |
| 4 | What the table does | Who, if anyone, gains, and what the rare states do for the applicants they describe |
| 5 | Decision | Adopt or leave out, and what that means for what comes next |

**A rule fixed now.** If stage 3 does not meet the four conditions, the table is left out. In that case there is no pruning, no change of settings, no calibration study and no submission. Pruning a list that adds nothing would be searching the folds for a subset that looks like a gain, which is exactly what the conditions exist to prevent. Stage 4 still looks at what the table does, because a table that adds nothing overall may still help a group, and that is worth knowing before the last table.

If stage 3 does meet the conditions, the notebook continues with the full sequence of notebook 08.

## Rules followed throughout

The same folds for every comparison; the holdout stays closed; Kaggle scores are never used to choose; everything that learns from the outcome sits inside the pipeline; a control with the new columns scrambled between applicants.

## Warnings carried in from notebook 09

- `POS_MAX_SHARE_LEFT` is distributed differently in train and test: test applicants are further through their running loans.
- The rare states (arrears at the application, a loan returned to the store, a loan past its term) concern a few hundred to a few thousand applicants.

## Stage 1: Setup, join and audit

No model is fitted in this stage.

In [1]:
import ast, hashlib, inspect, json, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import nbformat
import joblib
import lightgbm, sklearn, scipy
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

PROCESSED, RAW = '../data/processed', '../data/raw'
CACHE_DIR = Path('../artifacts/cv_cache_10')
sys.path.insert(0, '../src/models')
import pipeline as shared
from pipeline import SEED, N_SPLITS, CategoryEncoder, KeepColumns, CalibratedModel, lgbm_model, make_pipeline, cross_validate, compare, score_predictions, logit

# 1.1 the six pieces copied from notebook 02 are still identical to their source there
notebook_02 = nbformat.read('02_model_selection_training.ipynb', 4)
original = {}
for cell in notebook_02.cells:
    if cell.cell_type != 'code':
        continue
    try:
        tree = ast.parse(cell.source)
    except SyntaxError:
        continue
    for node in tree.body:
        if isinstance(node, (ast.ClassDef, ast.FunctionDef)):
            original.setdefault(node.name, []).append(ast.get_source_segment(cell.source, node))
COPIED = ['CategoryEncoder', 'KeepColumns', 'CalibratedModel', 'logit', 'score_predictions', 'describe']
fidelity = {name: len(original.get(name, [])) == 1 and original[name][0].strip() == inspect.getsource(getattr(shared, name)).strip() for name in COPIED}
print('identical to notebook 02:', fidelity)
assert all(fidelity.values())
MODULE_CHECKSUM = hashlib.md5(Path('../src/models/pipeline.py').read_bytes()).hexdigest()
print(f"module checksum {MODULE_CHECKSUM[:12]} (the module has not changed since notebook 04, which notebooks 06 and 08 also used, if this is 1379a001d7bb: {MODULE_CHECKSUM.startswith('1379a001d7bb')})")
print(f"results are cached in {CACHE_DIR.as_posix().replace('../', '')}")

identical to notebook 02: {'CategoryEncoder': True, 'KeepColumns': True, 'CalibratedModel': True, 'logit': True, 'score_predictions': True, 'describe': True}
module checksum 1379a001d7bb (the module has not changed since notebook 04, which notebooks 06 and 08 also used, if this is 1379a001d7bb: True)
results are cached in artifacts/cv_cache_10


In [2]:
recipe = json.loads(Path('../artifacts/model_with_installments/metadata.json').read_text(encoding='utf-8'))
CURRENT = list(recipe['features'])                                    # the 108 features of the current model
SOURCES = {k: list(v) for k, v in recipe['feature_sources'].items()}
BEST_PARAMS = dict(recipe['training']['lightgbm_settings'])
N_TREES, LEARNING_RATE = int(recipe['training']['trees']), float(recipe['training']['learning_rate'])
REFERENCE_CV = recipe['evidence']['cross_validation']
THRESHOLD = float(recipe['decision']['reference_threshold'])

def FINAL():
    return lgbm_model(BEST_PARAMS, N_TREES, LEARNING_RATE)

print(f"current model: {len(CURRENT)} features {({k: len(v) for k, v in SOURCES.items()})}, {N_TREES:,} trees, learning rate {LEARNING_RATE}, seed {recipe['training']['seed']}")
print(f"its cross-validated scores in notebook 08: ROC AUC {REFERENCE_CV['roc_auc']:.5f}, PR AUC {REFERENCE_CV['pr_auc']:.5f}, log loss {REFERENCE_CV['log_loss']:.5f} (after calibration)")
versions_now = {'python': sys.version.split()[0], 'lightgbm': lightgbm.__version__, 'pandas': pd.__version__, 'numpy': np.__version__, 'scikit-learn': sklearn.__version__, 'scipy': scipy.__version__}
assert recipe['versions'] == versions_now and recipe['training']['seed'] == SEED and len(CURRENT) == 108 and N_TREES == 3500
print('library versions as when the model was saved:', versions_now)

current model: 108 features {'application': 36, 'credit bureau': 24, 'previous applications': 25, 'instalment payments': 23}, 3,500 trees, learning rate 0.02, seed 42
its cross-validated scores in notebook 08: ROC AUC 0.79316, PR AUC 0.28483, log loss 0.23529 (after calibration)
library versions as when the model was saved: {'python': '3.13.5', 'lightgbm': '4.7.0', 'pandas': '3.0.6', 'numpy': '2.5.3', 'scikit-learn': '1.9.1', 'scipy': '1.18.1'}


In [3]:
split = pd.read_parquet(f'{PROCESSED}/application_split.parquet')
application = pd.read_parquet(f'{PROCESSED}/application_train_features.parquet')
application['REGION_RATING_CLIENT_W_CITY'] = application['REGION_RATING_CLIENT_W_CITY'].astype(float)
TABLES = ['bureau', 'previous', 'installments', 'pos_cash']
history = {name: pd.read_parquet(f'{PROCESSED}/{name}_train_features.parquet') for name in TABLES}
meta = {name: json.loads(Path(f'{PROCESSED}/{name}_features_metadata.json').read_text(encoding='utf-8')) for name in TABLES}
APP_ALL = [c for c in application.columns if c != 'TARGET']
POS = list(history['pos_cash'].columns)
TEXT_FEATURES = [c for c in APP_ALL if not pd.api.types.is_numeric_dtype(application[c])]
sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
all_columns = APP_ALL + [c for t in history.values() for c in t.columns]
join_checks = {
    'the new table: 307,511 rows, 12 features named POS_, all numeric': history['pos_cash'].shape == (307511, 12) and all(c.startswith('POS_') and pd.api.types.is_float_dtype(history['pos_cash'][c]) for c in POS),
    'each history file is the one its notebook saved (checksums)': all(sha(f'{PROCESSED}/{name}_train_features.parquet') == meta[name]['tables']['train']['sha256'] for name in TABLES),
    'the same applicants in the same order in all five': all(application.index.equals(t.index) for t in history.values()),
    'no column name in more than one table': len(all_columns) == len(set(all_columns)) == 55 + 40 + 41 + 29 + 12,
    'the 108 features of the current model are all present': set(CURRENT) <= set(all_columns),
    'the split covers exactly these applicants': split.index.equals(application.index),
}
for name, ok in join_checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
assert all(join_checks.values())

is_dev = (split['split'] == 'dev').values
joined = application.drop(columns='TARGET')
for t in history.values():
    joined = joined.join(t, how='left')
X_dev = joined.loc[is_dev].copy()
y_arr = application.loc[is_dev, 'TARGET'].values
dev_fold = split.loc[is_dev, 'fold'].to_numpy()
HOLDOUT_IDS = split.index[~is_dev]
del application, history, joined
FAMILY = {c: meta['pos_cash']['features'][c]['family'] for c in POS}
WARNED = {c: meta['pos_cash']['features'][c]['warning'] for c in POS if 'warning' in meta['pos_cash']['features'][c]}
short = lambda c: c.replace('POS_', '')
print(f"\ndevelopment set: {X_dev.shape[0]:,} applicants, {X_dev.shape[1]} features; defaults {int(y_arr.sum()):,} ({y_arr.mean():.2%}); holdout not loaded")
print(f"new features by family: {pd.Series(FAMILY).value_counts().to_dict()}; carrying a warning: {[short(c) for c in WARNED]}")
assert len(X_dev) == 246008 and int(y_arr.sum()) == 19860 and not X_dev.index.isin(HOLDOUT_IDS).any()

# the frozen folds, recomputed from the seed
fold_again = np.empty(len(X_dev), dtype=int)
for k, (_, val_idx) in enumerate(StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(len(y_arr)), y_arr)):
    fold_again[val_idx] = k
print(f"the folds recomputed from the seed are the frozen folds: {bool((fold_again == dev_fold).all())}")
assert (fold_again == dev_fold).all()

# leakage audit of the 12 new columns
numeric_old = [c for c in all_columns if c not in TEXT_FEATURES and c not in POS]
single = {}
for c in POS:
    ok = X_dev[c].notna().values
    a = roc_auc_score(y_arr[ok], X_dev[c].values[ok])
    single[c] = max(a, 1 - a)
single = pd.Series(single)
sample = X_dev.sample(60000, random_state=SEED)
cross = sample[numeric_old + POS].corr(method='spearman').loc[POS, numeric_old]
with_id = X_dev[POS].corrwith(pd.Series(X_dev.index.values, index=X_dev.index), method='spearman').abs()
audit = {
    'no new feature alone exceeds 0.65': bool((single <= 0.65).all()),
    'no new feature is a near-copy of an existing one (all under 0.95)': bool(cross.abs().max().max() < 0.95),
    'no new feature tracks the applicant ID (all under 0.05)': bool(with_id.max() < 0.05),
    'no outcome column, no holdout applicant': 'TARGET' not in X_dev.columns and not X_dev.index.isin(HOLDOUT_IDS).any(),
}
print(f"\nstrongest new feature alone: {short(single.idxmax())} {single.max():.4f}; closest existing feature to any new one: {cross.abs().max().max():.3f}")
for name, ok in audit.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
assert all(audit.values())

PASS  the new table: 307,511 rows, 12 features named POS_, all numeric
PASS  each history file is the one its notebook saved (checksums)
PASS  the same applicants in the same order in all five
PASS  no column name in more than one table
PASS  the 108 features of the current model are all present
PASS  the split covers exactly these applicants



development set: 246,008 applicants, 177 features; defaults 19,860 (8.07%); holdout not loaded
new features by family: {'arrears': 5, 'status': 3, 'commitment': 3, 'volume': 1}; carrying a warning: ['MAX_SHARE_LEFT']
the folds recomputed from the seed are the frozen folds: True



strongest new feature alone: MONTHS_SINCE_ARREARS 0.5956; closest existing feature to any new one: 0.836
PASS  no new feature alone exceeds 0.65
PASS  no new feature is a near-copy of an existing one (all under 0.95)
PASS  no new feature tracks the applicant ID (all under 0.05)
PASS  no outcome column, no holdout applicant


## Stage 2: Baseline

The current model is refitted from scratch and compared with notebook 08, applicant by applicant where notebook 08's saved predictions can be found. Then two other seeds measure the noise floor.

**Reproduced means**, fixed in advance: the mean ROC AUC within 0.0001 of notebook 08's and every fold within 0.0002.

In [4]:
NOTEBOOK_08_FOLDS = [0.79312, 0.78987, 0.79395, 0.79364, 0.79522]            # 'main model (108)' in notebook 08, section 9.4

def run(features, model=None, seed=None, X=None):
    model = FINAL() if model is None else model
    if seed is not None:
        model.set_params(random_state=seed)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return cross_validate(make_pipeline(features, TEXT_FEATURES, model), X_dev if X is None else X, y_arr, dev_fold, cache_dir=CACHE_DIR)

started = time.time()
baseline = run(CURRENT)
print(f"current model, {len(CURRENT)} features: {'loaded from cache' if baseline['cached'] else f'fitted in {(time.time() - started) / 60:.1f} minutes'}")
table = pd.DataFrame({'notebook 08': NOTEBOOK_08_FOLDS, 'rebuilt here': baseline['folds']['roc_auc'].values})
table['difference'] = table['rebuilt here'].round(5) - table['notebook 08']
print(table.to_string(float_format=lambda v: f'{v:.6f}'))
BASE_AUC = float(baseline['mean']['roc_auc'])
print(f"\nmean ROC AUC {BASE_AUC:.6f}; notebook 08 {REFERENCE_CV['roc_auc']:.6f}; difference {BASE_AUC - REFERENCE_CV['roc_auc']:+.6f}")
exact = None
for path in sorted(Path('../artifacts/cv_cache_08').glob('*.joblib')):
    try:
        item = joblib.load(path)
    except Exception:
        continue
    if isinstance(item, dict) and 'oof' in item and 'folds' in item and len(item['oof']) == len(y_arr) and np.allclose(item['folds']['roc_auc'].round(5).values, NOTEBOOK_08_FOLDS) \
            and abs(item['oof'].mean() - baseline['oof'].mean()) < 1e-6:
        gap = float(np.abs(item['oof'] - baseline['oof']).max())
        exact = gap if exact is None else min(exact, gap)
reproduced = {
    'mean ROC AUC within 0.0001 of notebook 08': abs(BASE_AUC - REFERENCE_CV['roc_auc']) < 0.0001,
    'every fold within 0.0002': bool((table['rebuilt here'] - table['notebook 08']).abs().max() < 0.0002),
    "identical to notebook 08's saved predictions, applicant by applicant": exact is not None and exact < 1e-12,
}
for name, ok in reproduced.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
assert reproduced['mean ROC AUC within 0.0001 of notebook 08'] and reproduced['every fold within 0.0002']

current model, 108 features: loaded from cache
   notebook 08  rebuilt here  difference
0     0.793120      0.793120    0.000000
1     0.789870      0.789874    0.000000
2     0.793950      0.793955    0.000000
3     0.793640      0.793638    0.000000
4     0.795220      0.795215    0.000000

mean ROC AUC 0.793160; notebook 08 0.793160; difference +0.000000


PASS  mean ROC AUC within 0.0001 of notebook 08
PASS  every fold within 0.0002
PASS  identical to notebook 08's saved predictions, applicant by applicant


In [5]:
SEEDS = [1, 2]
started = time.time()
seed_runs = {SEED: baseline, **{s: run(CURRENT, seed=s) for s in SEEDS}}
print(f"two more seeds: {(time.time() - started) / 60:.1f} minutes")
by_seed = pd.DataFrame({f'seed {s}': r['folds']['roc_auc'] for s, r in seed_runs.items()}).T
by_seed['mean'] = by_seed.mean(axis=1)
print(by_seed.to_string(float_format=lambda v: f'{v:.5f}'))
differences = pd.DataFrame({f'seed {s} minus seed {SEED}': compare(seed_runs[s], baseline) for s in SEEDS}).T
print()
print(differences[['mean_difference', 'standard_error', 'folds_better', 'ratio_to_standard_error']].astype({'folds_better': int}).to_string(float_format=lambda v: f'{v:+.5f}'))
SEED_RANGE = float(by_seed['mean'].max() - by_seed['mean'].min())
NOISE_FLOOR = round(max(0.0005, SEED_RANGE, float(differences['mean_difference'].abs().max())), 4)
print(f"\nrange of the mean across three seeds: {SEED_RANGE:.5f}; notebooks 04, 06 and 08 measured 0.0005 or less")
print(f"noise floor used from here on (the larger of the two): {NOISE_FLOOR:.4f}")

def verdict(result, reference=None):
    # the four conditions for an improvement, on the frozen folds
    c = compare(result, baseline if reference is None else reference)
    conditions = {'1. mean ROC AUC is higher': c['mean_difference'] > 0, '2. higher in at least four of five folds': c['folds_better'] >= 4,
                  '3. gain is more than twice its standard error': c['mean_difference'] > 0 and c['ratio_to_standard_error'] > 2,
                  f'4. gain exceeds the seed noise floor of {NOISE_FLOOR:.4f}': c['mean_difference'] > NOISE_FLOOR}
    return {**c, 'conditions': conditions, 'improves': all(conditions.values())}
assert not any(verdict(seed_runs[s])['improves'] for s in SEEDS)
print('no seed alone meets the four conditions:', not any(verdict(seed_runs[s])['improves'] for s in SEEDS))

two more seeds: 0.0 minutes
fold          0       1       2       3       4    mean
seed 42 0.79312 0.78987 0.79395 0.79364 0.79522 0.79316
seed 1  0.79302 0.79078 0.79432 0.79308 0.79562 0.79336
seed 2  0.79353 0.79021 0.79480 0.79389 0.79476 0.79344

                      mean_difference  standard_error  folds_better  ratio_to_standard_error
seed 1 minus seed 42         +0.00020        +0.00025             3                 +0.81101
seed 2 minus seed 42         +0.00028        +0.00021             4                 +1.31961

range of the mean across three seeds: 0.00028; notebooks 04, 06 and 08 measured 0.0005 or less
noise floor used from here on (the larger of the two): 0.0005
no seed alone meets the four conditions: True


## Stage 3: Add the new features

The model of stage 2, given the 12 new features as well as its 108. Same settings, trees, seed and folds. With the scrambled control and two other seeds.

In [6]:
WITH_NEW = CURRENT + POS
assert len(WITH_NEW) == 120 and len(set(WITH_NEW)) == 120
started = time.time()
with_new = run(WITH_NEW)
print(f"current model + monthly balances, {len(WITH_NEW)} features: {'loaded from cache' if with_new['cached'] else f'fitted in {(time.time() - started) / 60:.1f} minutes'}")
folds = pd.DataFrame({'current model (108)': baseline['folds']['roc_auc'], 'with the monthly balances (120)': with_new['folds']['roc_auc']})
folds['gain'] = folds.iloc[:, 1] - folds.iloc[:, 0]
print(folds.to_string(float_format=lambda v: f'{v:.5f}'))
headline = verdict(with_new)
GAIN = headline['mean_difference']
print(f"\nmean ROC AUC: {BASE_AUC:.5f} -> {with_new['mean']['roc_auc']:.5f}   gain {GAIN:+.5f}; standard error {headline['standard_error']:.5f}; better in {headline['folds_better']} of 5 folds")
for name, ok in headline['conditions'].items():
    print(f"{'MET    ' if ok else 'NOT MET'}  {name}")
ADOPT = bool(headline['improves'])
print(f"\nverdict: the new features {'IMPROVE' if ADOPT else 'DO NOT IMPROVE'} the model by the conditions fixed in advance")
print(f"PR AUC: {baseline['mean']['pr_auc']:.5f} -> {with_new['mean']['pr_auc']:.5f} (better in {int((with_new['folds']['pr_auc'] > baseline['folds']['pr_auc']).sum())} of 5 folds)")

current model + monthly balances, 120 features: loaded from cache
      current model (108)  with the monthly balances (120)     gain
fold                                                               
0                 0.79312                          0.79334  0.00022
1                 0.78987                          0.79041  0.00054
2                 0.79395                          0.79368 -0.00027
3                 0.79364                          0.79371  0.00007
4                 0.79522                          0.79501 -0.00020

mean ROC AUC: 0.79316 -> 0.79323   gain +0.00007; standard error 0.00015; better in 3 of 5 folds
MET      1. mean ROC AUC is higher
NOT MET  2. higher in at least four of five folds
NOT MET  3. gain is more than twice its standard error
NOT MET  4. gain exceeds the seed noise floor of 0.0005

verdict: the new features DO NOT IMPROVE the model by the conditions fixed in advance
PR AUC: 0.28483 -> 0.28527 (better in 4 of 5 folds)


In [7]:
rng = np.random.default_rng(SEED)
X_scrambled = X_dev.copy()
X_scrambled[POS] = X_dev[POS].to_numpy()[rng.permutation(len(X_dev))]
assert X_scrambled[CURRENT].equals(X_dev[CURRENT])
started = time.time()
scrambled = run(WITH_NEW, X=X_scrambled)
del X_scrambled
paired = {s: run(WITH_NEW, seed=s) for s in SEEDS}
print(f"control and two seeds: {(time.time() - started) / 60:.1f} minutes")
rows = {'real features, seed 42': verdict(with_new), 'scrambled features, seed 42': verdict(scrambled),
        'real features, seed 1 (against seed 1)': verdict(paired[1], seed_runs[1]), 'real features, seed 2 (against seed 2)': verdict(paired[2], seed_runs[2])}
controls = pd.DataFrame({label: {'mean_gain': v['mean_difference'], 'standard_error': v['standard_error'], 'folds_better': v['folds_better'], 'meets_all_four': v['improves']} for label, v in rows.items()}).T
print(controls.to_string(float_format=lambda v: f'{v:+.5f}'))
real = [rows[k]['mean_difference'] for k in rows if k.startswith('real')]
print(f"\ngain with real features across three seeds: {min(real):+.5f} to {max(real):+.5f} (mean {np.mean(real):+.5f}); with scrambled features {rows['scrambled features, seed 42']['mean_difference']:+.5f}")
print(f"seeds on which the real features meet all four conditions: {sum(rows[k]['improves'] for k in rows if k.startswith('real'))} of 3")

control and two seeds: 0.1 minutes
                                       mean_gain standard_error folds_better meets_all_four
real features, seed 42                  +0.00007       +0.00015            3          False
scrambled features, seed 42             +0.00003       +0.00015            3          False
real features, seed 1 (against seed 1)  -0.00016       +0.00021            2          False
real features, seed 2 (against seed 2)  +0.00001       +0.00028            2          False

gain with real features across three seeds: -0.00016 to +0.00007 (mean -0.00003); with scrambled features +0.00003
seeds on which the real features meet all four conditions: 0 of 3


### Stage 1 to 3 findings

**Setup and baseline**
- The five tables join exactly, each history file matches the checksum its notebook recorded, and the folds recomputed from the seed are the frozen ones. The 12 new columns pass the leakage audit: none tracks the applicant ID, the strongest alone scores 0.596, and none is a near-copy of an existing feature (the closest is 0.84).
- The 108-feature model rebuilt here scores 0.79316, every fold equal to notebook 08's to five decimals, and its predictions equal the ones notebook 08 saved for every applicant.
- Two other seeds move the mean by 0.0002 and 0.0003. The noise floor stays at 0.0005.

**The monthly balances do not improve the model**

| Fold | Current model (108) | With the monthly balances (120) | Gain |
|---|---:|---:|---:|
| 0 | 0.79312 | 0.79334 | +0.00022 |
| 1 | 0.78987 | 0.79041 | +0.00054 |
| 2 | 0.79395 | 0.79368 | -0.00027 |
| 3 | 0.79364 | 0.79371 | +0.00007 |
| 4 | 0.79522 | 0.79501 | -0.00020 |
| **Mean** | **0.79316** | **0.79323** | **+0.00007** |

- The mean is 0.00007 higher, in 3 of 5 folds, with a standard error of 0.00015. Three of the four conditions are not met.
- **The real features do no better than scrambled ones.** Given another applicant's monthly balances, the model scores +0.00003, against +0.00007 with the real ones. In notebook 08 the real features beat the scrambled control by 0.0085.
- On the two other seeds the gain is -0.00016 and +0.00001. Over three seeds the mean is -0.00003.
- PR AUC is 0.0004 higher, in 4 of 5 folds. That is within what the seeds move it by and is not one of the conditions.
- This is what notebook 09 expected, in writing, before this notebook was created: a gain at or below the noise floor.

**Decision, by the rule fixed in the introduction**
- The table is left out. There is no pruning, no change of settings, no calibration study and no submission. Stage 4 looks at whether any group of applicants gains, using the predictions already made, and fits nothing new.

## Stage 4: What the table does

No new model is fitted. The predictions of stage 3, with and without the new features, are compared for the groups notebook 09 pointed to.

For each group: its ROC AUC within the group, averaged over the folds, and its average position in the ordering of all applicants by risk (1 is the riskiest), under each model.

In [8]:
inst = lambda c: X_dev['INST_' + c]
pos_ = lambda c: X_dev['POS_' + c]
GROUPS = {
    'all applicants': np.ones(len(X_dev), dtype=bool),
    'no monthly rows': X_dev[POS].isna().all(axis=1).values,
    'in arrears at the application': (pos_('N_ARREARS_AT_APPLICATION') > 0).values,
    'in arrears in the last 12 months': (pos_('MONTHS_ARREARS_12M') > 0).values,
    'ever in arrears': (pos_('N_LOANS_ARREARS') > 0).values,
    'a loan returned to the store': (pos_('N_RETURNED') > 0).values,
    'a running loan past its term': (pos_('N_PAST_TERM') > 0).values,
    'a running loan with over half its term to go': (pos_('MAX_SHARE_LEFT') > 0.5).values,
    'an unpaid instalment (under-predicted in notebook 08)': ((inst('N_UNPAID_RECENT') + inst('N_UNPAID_OLDER')) > 0).values,
    '   of these, in arrears at the application': ((inst('N_UNPAID_RECENT') + inst('N_UNPAID_OLDER')) > 0).values & (pos_('N_ARREARS_AT_APPLICATION') > 0).values,
}
rank_old, rank_new = pd.Series(baseline['oof']).rank(pct=True).values, pd.Series(with_new['oof']).rank(pct=True).values
def within(result, m):
    scores = [roc_auc_score(y_arr[m & (dev_fold == k)], result['oof'][m & (dev_fold == k)]) for k in range(N_SPLITS) if 0 < y_arr[m & (dev_fold == k)].sum() < (m & (dev_fold == k)).sum()]
    return (float(np.mean(scores)), len(scores)) if scores else (np.nan, 0)
rows = {}
for label, m in GROUPS.items():
    (a_old, k_old), (a_new, _) = within(baseline, m), within(with_new, m)
    rows[label] = {'applicants': int(m.sum()), 'default_pct': y_arr[m].mean() * 100, 'auc_current': a_old, 'auc_with_new': a_new, 'change': a_new - a_old, 'folds_scored': k_old,
                   'position_current': rank_old[m].mean(), 'position_with_new': rank_new[m].mean()}
who = pd.DataFrame(rows).T
who['position_change'] = who['position_with_new'] - who['position_current']
print(who.astype({'applicants': int, 'folds_scored': int}).to_string(float_format=lambda v: f'{v:,.4f}'))

declined_old = baseline['oof'] >= np.quantile(baseline['oof'], 0.714)
declined_new = with_new['oof'] >= np.quantile(with_new['oof'], 0.714)
changed = declined_old != declined_new
print(f"\ndeclining the same 28.6% of applicants, decisions that change: {changed.mean():.2%}; newly declined default at {y_arr[declined_new & ~declined_old].mean():.1%}, newly approved at {y_arr[~declined_new & declined_old].mean():.1%}")
print(f"for comparison, two seeds of the current model change {(declined_old != (seed_runs[1]['oof'] >= np.quantile(seed_runs[1]['oof'], 0.714))).mean():.2%} of decisions")

                                                       applicants  default_pct  auc_current  auc_with_new  change  folds_scored  position_current  position_with_new  position_change
all applicants                                             246008       8.0729       0.7932        0.7932  0.0001             5            0.5000             0.5000           0.0000
no monthly rows                                             14452       6.6842       0.8160        0.8162  0.0002             5            0.4413             0.4412          -0.0002
in arrears at the application                                 685      22.1898       0.7856        0.7925  0.0069             5            0.7297             0.7602           0.0305
in arrears in the last 12 months                             5644      15.6449       0.7739        0.7737 -0.0002             5            0.6804             0.7014           0.0210
ever in arrears                                             31991      10.7780       0.783

## Stage 5: Decision

In [9]:
summary = pd.DataFrame({
    'current model (notebook 08)': {'features': len(CURRENT), 'roc_auc': BASE_AUC, 'pr_auc': float(baseline['mean']['pr_auc'])},
    'with the monthly balances': {'features': len(WITH_NEW), 'roc_auc': float(with_new['mean']['roc_auc']), 'pr_auc': float(with_new['mean']['pr_auc'])},
    'with the monthly balances, scrambled': {'features': len(WITH_NEW), 'roc_auc': float(scrambled['mean']['roc_auc']), 'pr_auc': float(scrambled['mean']['pr_auc'])},
}).T
print(summary.astype({'features': int}).to_string(float_format=lambda v: f'{v:.5f}'))
print(f"\nadopted: {ADOPT}. The model carried forward is notebook 08's, unchanged: artifacts/model_with_installments/")
assert not ADOPT

                                      features  roc_auc  pr_auc
current model (notebook 08)                108  0.79316 0.28483
with the monthly balances                  120  0.79323 0.28527
with the monthly balances, scrambled       120  0.79319 0.28494

adopted: False. The model carried forward is notebook 08's, unchanged: artifacts/model_with_installments/


### Stage 4 and 5 findings, and conclusions

**Some small groups move, and the decisions do not get better**

| Group | Applicants | Default rate | Change in ROC AUC within the group | Position, current model | With the monthly balances |
|---|---:|---:|---:|---:|---:|
| In arrears at the application | 685 | 22.2% | +0.0069 | 0.730 | 0.760 |
| In arrears in the last 12 months | 5,644 | 15.6% | -0.0002 | 0.680 | 0.701 |
| A loan returned to the store | 3,422 | 10.3% | +0.0016 | 0.531 | 0.557 |
| A running loan past its term | 3,837 | 6.6% | -0.0025 | 0.488 | 0.447 |
| Ever in arrears | 31,991 | 10.8% | -0.0003 | 0.582 | 0.587 |
| No monthly rows | 14,452 | 6.7% | +0.0002 | 0.441 | 0.441 |
| An unpaid instalment | 397 | 27.2% | -0.0013 | 0.721 | 0.722 |
| ... of these, in arrears at the application | 91 | 56.0% | | 0.941 | 0.952 |

- The rare states do what rare-event features do. Applicants in arrears at the application move up the risk ordering, from 0.73 to 0.76; those with a loan past its term but otherwise paying move down, which fits their low default rate. The groups are small, and the overall score does not register them.
- But the decisions do not improve. Declining the same 28.6% of applicants, 2.9% of decisions change, against 2.6% for a change of random seed alone, and those newly declined default at 9.0% while those newly approved default at 9.1%. The model swaps applicants of the same risk.
- **I had the group of 91 wrong.** Notebook 09 found that the applicants with an unpaid instalment who were in arrears at the application default at about 55%, and I expected this table to let the model reach them. The current model already places them at 0.94, among the riskiest 6% of all applicants. Its problem with them, found in notebook 08, is the level of the probability it gives them, not their place in the ordering. Monthly balances do not change that: they move them from 0.94 to 0.95.
- So notebook 08's explanation of the under-prediction, the minimum leaf size, is about calibration within a tiny group and not about ranking. That makes it less important than I suggested there: for decisions at the reference threshold, these applicants are declined either way.

**The decision**
- **The monthly balances are left out.** The model of notebook 08 is carried forward unchanged: 108 features, cross-validated ROC AUC 0.7932, Kaggle private 0.7934.
- Nothing was pruned, tuned or submitted, by the rule fixed in the introduction.
- `data/processed/pos_cash_*_features.parquet` and `src/features/pos_cash.py` stay on disk with their metadata. They are verified, and can be tested again if the settings change.

**What this table showed**
- **A screen can say "nothing new" correctly.** Notebook 09's screen found almost nothing beyond the current model and wrote down, before this notebook existed, that it expected a gain at or below the noise floor. The result is +0.00007, with the scrambled control at +0.00003. Over five tables the one-feature-at-a-time screen has been right three times and wrong twice: it under-predicted the bureau gain (notebook 04) and over-predicted refusals (notebook 06).
- **Overlap, not quality, is the reason.** The monthly balances are a sound record of the same loans that the payment table and `previous_application` already describe.

**Leakage check**
- The baseline reproduced notebook 08's predictions exactly. Every comparison was on the frozen folds with unchanged settings. Nothing was selected. The holdout was not used.

**What comes next**
1. `credit_card_balance`, the last table, by the same method: notebook 11 for its features and notebook 12 to test them.
2. A search of the settings on the final feature list, inside the folds. Notebook 08 found both targeted checks leaning towards more trees and fewer features per tree.
3. The holdout, opened once, for the final model.